# M2-B2 — Biais du modèle d'annulation (binôme)

Le calcul est fourni ; les constats, les choix et leur interprétation sont à rédiger dans les cellules ✏️.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
from features import INPUT_FEATURES
from modele import entrainer, mesures_par_groupe

train = pd.read_csv(ROOT / "data/reservations_train.csv")
test = pd.read_csv(ROOT / "data/reservations_test.csv")
v21 = joblib.load(ROOT / "model/cancellation_model_v2.1.0.joblib")
len(train), len(test)

(58199, 19694)

## Tâche 2 — Mesurer

Groupes : clients portugais (`PRT`) et autres pays (pays manquant compris).

In [24]:
groupe = test["country"].eq("PRT").map({True: "Portugal", False: "Autres pays"})
mesures_par_groupe(v21, test, INPUT_FEATURES, groupe)

,effectif,annulations_reelles,taux_alertes,rappel,fausses_alertes,precision
Autres pays,12871.0,0.258,0.223,0.530,0.117,0.612
Portugal,6823.0,0.620,0.744,0.931,0.438,0.776


### ✏️ Constats

Les portugais pesent 34,6% des effectif et annule à eux seul 62% contre 25% pour le reste des autres pays.
Les portugais representent 43% des fausses alerte, ce qui veut dire qu'ils recoivent le plus de souvent une alerte à tort.


## Tâche 3 — Débattre

| | Arguments |
|---|---|
| Position A : l'écart est acceptable | le modèle mesure un risque, pas une nationalité. 62 % contre 25,8 % d’annulations réelles. Égaliser les alertes = chambres vides et relances inutiles. Chiffre : rappel et precision comparables entre groupes. |
| Position B : l'écart est injuste | le préjudice touche ceux qui viennent quand même. Chiffre : fausses_alertes. Le pays est subi, pas choisi. Effet de boucle au réentraînement. Géoblocage 2018/302.  |
| Décision du binôme, si l'alerte déclenche une relance | : on garde, sous conditions. Message identique, pays caché à l’agent, écart de fausses_alertes suivi à chaque version. |
| Décision du binôme, si l'alerte déclenche un acompte | non en l’état. Soit uniforme pour tous, soit sans country avec revue humaine (art. 22). |

## Tâche 4 — Atténuer

`entrainer(train, colonnes)` réentraîne la même recette que la v2.1 (seuil 0,2) sur les colonnes données.

In [25]:
colonnes_sans_pays = [c for c in INPUT_FEATURES if c != "country"]
print("Colonnes sans country :", colonnes_sans_pays)
v21_sans_pays = entrainer(train, colonnes_sans_pays)
mesures_par_groupe(v21_sans_pays, test, colonnes_sans_pays, test["country"].eq("PRT").map({True: "Portugal", False: "Autres pays"})) 


Colonnes sans country : ['hotel', 'arrival_date_month', 'market_segment', 'deposit_type', 'customer_type', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'adr', 'previous_cancellations', 'total_of_special_requests', 'required_car_parking_spaces']


,effectif,annulations_reelles,taux_alertes,rappel,fausses_alertes,precision
Autres pays,12871.0,0.258,0.438,0.695,0.348,0.409
Portugal,6823.0,0.620,0.551,0.776,0.184,0.873


### À investiguer

Peut-on deviner le pays sans la colonne `country` ? Choisir une méthode pour répondre, et dire quelles variables sont en cause.

Méthode : même recette, mêmes colonnes sans `country`, mais cible « le client est portugais ». Lecture à l'AUC, pas au rappel : au seuil 0,2 le modèle classe 97,6 % des réservations comme portugaises.

Résultat : AUC 0,838, le pays est déductible.

Variables en cause :
- `deposit_type` : « Non Refund » portugais à 100 %, « No Deposit » 25 %, « Refundable » 10 %
- `market_segment` : « Complementary » 90 %, « Corporate » 77 %, contre « Online TA » 15 %
- `lead_time` 155 jours contre 105, demandes spéciales 0,33 contre 0,80

Retirer `country` retire l'étiquette, pas l'information. Et `deposit_type` n'est pas retirable sans perdre le signal le plus fort du modèle.

In [26]:
from sklearn.metrics import roc_auc_score

# Cible : « le client est portugais », sans la colonne country.
train_pays = train.assign(is_canceled=train["country"].eq("PRT").astype(int))
test_pays = test.assign(is_canceled=test["country"].eq("PRT").astype(int))
devine_pays = entrainer(train_pays, colonnes_sans_pays)

# Le seuil 0,2 est cale sur l'annulation : c'est l'AUC qui repond.
proba = devine_pays.predict_proba(test_pays[colonnes_sans_pays])[:, 1]
print("AUC :", round(roc_auc_score(test_pays["is_canceled"], proba), 3))

noms = devine_pays.named_steps["preprocessing"].get_feature_names_out()
coefs = devine_pays.named_steps["classifier"].estimator_.coef_[0]
pd.Series(coefs, index=noms).sort_values(key=abs, ascending=False).head(12).round(2)

AUC : 0.838


cat__deposit_type_Non Refund         2.55
cat__market_segment_Complementary    1.98
cat__deposit_type_Refundable        -1.57
cat__market_segment_Online TA       -1.01
cat__market_segment_Offline TA/TO   -0.59
cat__market_segment_Corporate        0.55
cat__deposit_type_No Deposit        -0.53
cat__hotel_Resort Hotel              0.52
num__previous_cancellations          0.43
cat__arrival_date_month_July         0.39
cat__market_segment_Groups          -0.37
cat__arrival_date_month_April       -0.34
dtype: float64

### ✏️ Interprétation

1. L'écart ne disparaît pas, il change de sens. Sur les fausses alertes : 43,8 % contre 11,7 % avec `country`, 18,4 % contre 34,8 % sans. Même inversion sur le rappel (93,1 → 77,6 côté Portugal, 53,0 → 69,5 ailleurs) et sur la précision. Seul le taux d'annulation réel ne bouge pas : c'est une donnée du jeu de test, pas une sortie du modèle.
2. Les clients portugais gagnent sur les fausses alertes et perdent en détection. Les clients étrangers l'inverse : leurs fausses alertes triplent et leur précision tombe de 61,2 % à 40,9 %. Le retrait de la variable déplace le préjudice, il ne le supprime pas.
3. Le coût est élevé : précision globale 71,7 % → 59,5 % pour un rappel inchangé (75,5 % → 74,1 %). Autrement dit, on alerte sur 47,7 % des réservations au lieu de 40,4 % sans détecter une annulation de plus. Non acceptable en l'état.
4. Non, aucune variable supplémentaire à collecter. Le pays se retrouve sans la colonne : AUC 0,838, et `deposit_type = Non Refund` est portugais à 100 %. Retirer `country` ne retire pas l'information, donc la réponse n'est pas dans les données mais dans l'usage : encadrer ce que l'alerte déclenche, et suivre l'écart de fausses alertes par groupe à chaque réentraînement.